# Field distribution comparison

Compare PDFs and joint PDFs of glider-sampled fields against the model field in
the array hull, across array configurations.

In [ ]:
import json
import os
import sys
import numpy as np

os.chdir('/home/edavenport/analysis/tpose24-osse')
sys.path.insert(0, '/home/edavenport/analysis/tpose24-osse')
from osse_tools import (load_model, load_positions, sample_fields, model_region,
                        add_density, eddy_anomalies, compute_w_planefit,
                        model_divergence, dist_stats, plot_field_pdfs,
                        plot_joint_compare, plot_pdf_compare)
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
RUN_DIR   = '/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons'
ITERS     = list(range(36, 26173, 36))
MAX_DEPTH = 70
TSTEP     = 4          # subsample time when pooling the regional truth
BASE_OUTDIR = 'field_pdfs/70m'

CONFIG_FILES = [
    'configs/rect_x0.25.json',
    'configs/rect_x0.5.json',
    'configs/rect_x1.0.json',
]

In [ ]:
ds = load_model(RUN_DIR, ITERS)
ds = ds.sel(time=slice('2012-10-11', None))  # exclude spin-up

## Sample observed (gliders) and true (hull) populations

In [ ]:
results = {}

for cfg_file in CONFIG_FILES:
    with open(cfg_file) as f:
        cfg = json.load(f)
    positions = load_positions(cfg_file)
    dss = ds.isel(time=slice(None, None, TSTEP))

    obs  = eddy_anomalies(add_density(sample_fields(dss, positions, max_depth=MAX_DEPTH))).compute()
    true = eddy_anomalies(add_density(model_region(dss, positions, max_depth=MAX_DEPTH))).compute()
    results[cfg['name']] = dict(positions=positions, obs=obs, true=true)
    print(cfg['name'], 'obs', dict(obs.sizes), 'true', dict(true.sizes))

## 1-D field PDFs and eddy joint PDFs per configuration

In [ ]:
for name, r in results.items():
    outdir = f'{BASE_OUTDIR}/{name}'
    os.makedirs(outdir, exist_ok=True)
    obs, true = r['obs'], r['true']

    fig = plot_field_pdfs(obs, true)
    fig.suptitle(f'{name}: field PDFs  (0-{MAX_DEPTH} m)', y=1.01)
    fig.savefig(f'{outdir}/field_pdfs.png', dpi=150, bbox_inches='tight')
    plt.show()

    pairs = [
        (obs.U,  obs.V,  true.U,  true.V,  ('U', 'V'),    ('m/s', 'm/s'),   'uv'),
        (obs.T,  obs.S,  true.T,  true.S,  ('T', 'S'),    ('°C', 'g/kg'),   'TS'),
        (obs.Up, obs.Vp, true.Up, true.Vp, ("u'", "v'"),  ('m/s', 'm/s'),   'stress_uv'),
        (obs.Vp, obs.Tp, true.Vp, true.Tp, ("v'", "T'"),  ('m/s', '°C'),    'heatflux_vT'),
        (obs.Up, obs.Tp, true.Up, true.Tp, ("u'", "T'"),  ('m/s', '°C'),    'heatflux_uT'),
    ]
    for ox, oy, tx, ty, labels, units, tag in pairs:
        fig = plot_joint_compare(ox, oy, tx, ty, labels, units)
        fig.suptitle(f'{name}: {labels[0]}-{labels[1]}', y=1.02)
        fig.savefig(f'{outdir}/joint_{tag}.png', dpi=150, bbox_inches='tight')
        plt.show()

## Divergence: plane-fit estimate vs true area mean

In [ ]:
for name, r in results.items():
    positions = r['positions']
    div_obs  = compute_w_planefit(sample_fields(ds, positions, vars=('UVEL', 'VVEL'),
                                                max_depth=MAX_DEPTH))['div']
    div_true = model_divergence(ds, positions, max_depth=MAX_DEPTH)
    fig = plot_pdf_compare(div_obs, div_true, label='divergence', units='1/s')
    fig.suptitle(f'{name}: horizontal divergence', y=1.02)
    plt.show()

## Summary table of observed-vs-true distribution distance

In [ ]:
hdr = f"{'config':>12}  {'field':>7}  {'KS':>6}  {'Wass':>10}"
print(hdr); print('-' * len(hdr))
for name, r in results.items():
    for v in ('T', 'S', 'U', 'V', 'sigma0'):
        s = dist_stats(r['obs'][v], r['true'][v])
        print(f"{name:>12}  {v:>7}  {s['ks']:>6.2f}  {s['wasserstein']:>10.2e}")